## Exercises on Reinforcement Learning for Language Models

These paper-and-pencil exercises reinforce Chapter 13: the language model as a policy over tokens, the Bradley–Terry reward model and its loss, the KL-regularised objective and its per-token decomposition, the closed-form optimal policy and the DPO derivation, the DPO gradient weights, group-relative advantages, the per-token KL estimator of GRPO, and reward over-optimisation. Notation: prompt $x$, response $y=(y_1,\dots,y_T)$, policy $\pi_\theta$, reference $\pi_{\text{ref}}$, reward model $r_\phi$, KL coefficient $\beta$, preferred/rejected responses $y_w, y_l$.

### Exercise 13.1 — A response as an episode

A model with a vocabulary of $|\mathcal V|=4$ tokens $\{a,b,c,\langle\text{eos}\rangle\}$ answers a prompt with the response $y=(a,\,b,\,\langle\text{eos}\rangle)$. Its next-token probabilities along the way are $\pi(a\mid x)=0.5$, $\pi(b\mid x,a)=0.4$, $\pi(\langle\text{eos}\rangle\mid x,a,b)=0.8$.

1. Compute $\pi_\theta(y\mid x)$ and $\log\pi_\theta(y\mid x)$.
2. Identify states, actions and rewards of this episode if the reward model gives $r_\phi(x,y)=2$.
3. How many distinct responses of at most $3$ tokens exist, counting both the responses that end with the end token and those truncated by the length limit without it? Comment on the size of the "action space" of a real model.

**Step 1 — Sequence probability.** By the chain rule, $\ \pi_\theta(y\mid x)=0.5\times0.4\times0.8 = 0.16$, and $\log\pi_\theta(y\mid x)=\ln0.5+\ln0.4+\ln0.8 = -0.693-0.916-0.223 = -1.833$.

**Step 2 — The episode.** States: $S_1=x$, $S_2=(x,a)$, $S_3=(x,a,b)$; actions: $A_1=a$, $A_2=b$, $A_3=\langle\text{eos}\rangle$; transitions deterministic (append the token); rewards $R_2=R_3=0$ and $R_4=r_\phi(x,y)=2$ at the end of the response. The return of every step is $2$ (no discount), so the policy gradient weights all three token log-probabilities by the same number — which is exactly what the sequence-level advantage of the chapter's implementation does.

**Step 3 — Counting.** Responses end at the first $\langle\text{eos}\rangle$: length $1$: $1$ response; length $2$: $3$ ($3$ choices then eos); length $3$: $3^2=9$ (two free tokens then eos) plus $3^3=27$ responses of three free tokens cut off by the length limit. In total $1+3+9+27=40$. For a real model with $|\mathcal V|\approx10^5$ and responses of hundreds of tokens, the number of possible episodes is astronomically large and the reward is a single number per episode: the credit-assignment problem of Chapter 02 in its most extreme form.

**Key concept**

A response is a trajectory in a deterministic MDP whose actions are tokens; its log-probability is the sum of the token log-probabilities, and the single terminal reward is shared by every token when the advantage is computed at the sequence level.

### Exercise 13.2 — The Bradley–Terry reward model

A reward model currently scores three responses to a prompt as $r(y_1)=1.0$, $r(y_2)=1.5$, $r(y_3)=3.0$.

1. Compute $P(y_2\succ y_1)$, $P(y_3\succ y_1)$ and $P(y_3\succ y_2)$ under the Bradley–Terry model.
2. An annotator reports $y_1\succ y_2$. Compute the loss $-\log\sigma\big(r(y_w)-r(y_l)\big)$ of this comparison and the gradient of the loss with respect to $r(y_1)$ and $r(y_2)$.
3. Show that adding a constant $c$ to all scores leaves every probability unchanged, and explain what this implies for the reward used in RLHF.

**Step 1 — Probabilities.** $\ P(y_2\succ y_1)=\sigma(0.5)=0.622$; $\ P(y_3\succ y_1)=\sigma(2.0)=0.881$; $\ P(y_3\succ y_2)=\sigma(1.5)=0.818$. Differences of a few units give confident preferences; a difference of $0.5$ is close to a coin flip.

**Step 2 — Loss and gradient.** With $y_w=y_1$, $y_l=y_2$: $\ \mathcal L=-\log\sigma(1.0-1.5)=-\log\sigma(-0.5)=-\log0.378 = 0.974$. Since $\frac{d}{dz}\big[-\log\sigma(z)\big]=-(1-\sigma(z))$, with $z=r(y_1)-r(y_2)=-0.5$ and $1-\sigma(-0.5)=0.622$:

$\displaystyle \frac{\partial\mathcal L}{\partial r(y_1)} = -0.622, \qquad \frac{\partial\mathcal L}{\partial r(y_2)} = +0.622.$

Gradient descent raises the score of the preferred response and lowers that of the rejected one, by an amount equal to the probability the model currently assigns to the *wrong* ordering: surprising comparisons move the model most.

**Step 3 — Invariance.** $\ \sigma\big((r(y_w)+c)-(r(y_l)+c)\big)=\sigma\big(r(y_w)-r(y_l)\big)$: the comparisons determine the reward only up to an additive constant per prompt. In RLHF this constant is harmless for the policy gradient (it is a baseline, Chapter 08), but it means that absolute reward values across prompts are not comparable, which is one reason to standardise rewards within a batch or a group.

**Key concept**

The reward model is a logistic regression on score differences: its gradient on a comparison is the probability of the wrong ordering, and its scores are defined only up to a per-prompt constant.

### Exercise 13.3 — The KL penalty token by token

For the response of Exercise 13.1 the reference model has $\pi_{\text{ref}}(a\mid x)=0.25$, $\pi_{\text{ref}}(b\mid x,a)=0.4$, $\pi_{\text{ref}}(\langle\text{eos}\rangle\mid x,a,b)=0.2$, and $\beta=0.5$.

1. Compute the per-token log-ratios $\log\frac{\pi_\theta(y_t\mid\cdot)}{\pi_{\text{ref}}(y_t\mid\cdot)}$ and their sum.
2. Write the per-token rewards $R_t$ of the KL-regularised problem (reward model score $2$ at the end) and the total reward of the response.
3. Explain why the sum of log-ratios over sampled responses estimates $D_{\text{KL}}(\pi_\theta\|\pi_{\text{ref}})$, and why the estimate can be negative for a single response.

**Step 1 — Log-ratios.** $\ \ln\frac{0.5}{0.25}=0.693$, $\ \ln\frac{0.4}{0.4}=0$, $\ \ln\frac{0.8}{0.2}=1.386$; sum $2.079$. The policy has made this response $e^{2.079}=8$ times more likely than the reference did.

**Step 2 — Per-token rewards.** With the convention of the course ($R_{t+1}$ rewards the action $A_t=y_t$): $R_{t+1}=-\beta\,(\text{log-ratio})_t$ for $t<T$ and $R_{T+1}=r_\phi-\beta\,(\text{log-ratio})_T$:

$\displaystyle R_2=-0.5(0.693)=-0.347,\qquad R_3=0,\qquad R_4=2-0.5(1.386)=1.307,$

total $2-0.5(2.079)=0.961$. The sequence-level advantage of the implementation is this total minus the batch mean, rescaled.

**Step 3 — Estimator.** $D_{\text{KL}}(\pi_\theta\|\pi_{\text{ref}})=\mathbb E_{y\sim\pi_\theta}\big[\log\pi_\theta(y\mid x)-\log\pi_{\text{ref}}(y\mid x)\big]$, and the log-ratio of a sequence is the sum of its token log-ratios, so the average of the summed log-ratios over responses *sampled from $\pi_\theta$* is an unbiased estimate. For one response the log-ratio can be negative (a response that the policy makes *less* likely than the reference), although the expectation is non-negative; the non-negative estimator $\frac{\pi_{\text{ref}}}{\pi_\theta}-\log\frac{\pi_{\text{ref}}}{\pi_\theta}-1$ of GRPO avoids this.

**Key concept**

Because sequence log-probabilities add over tokens, the KL penalty of RLHF can be charged as a per-token negative reward, turning the regularised objective into an ordinary episodic problem; the terminal step carries the reward model's score.

### Exercise 13.4 — The optimal policy of the KL-regularised objective

A prompt admits only three responses, with reference probabilities $\pi_{\text{ref}}=(0.5,\,0.3,\,0.2)$ and rewards $r=(0,\,1,\,2)$.

1. Compute the optimal policy $\pi^*(y)\propto\pi_{\text{ref}}(y)\,e^{r(y)/\beta}$ for $\beta=1$ and for $\beta=0.25$.
2. Compute, for $\beta=1$, the expected reward and the KL divergence from the reference of $\pi^*$, and the value of the objective $\mathbb E[r]-\beta D_{\text{KL}}$.
3. What are the limits $\beta\to\infty$ and $\beta\to0$?

**Step 1 — Optimal policies.** $\beta=1$: unnormalised weights $0.5e^0,\ 0.3e^1,\ 0.2e^2 = 0.5,\ 0.815,\ 1.478$; $Z=2.793$; $\pi^*=(0.179,\,0.292,\,0.529)$.

$\beta=0.25$: weights $0.5e^0,\ 0.3e^4,\ 0.2e^8 = 0.5,\ 16.38,\ 596.2$; $Z=613.1$; $\pi^*=(0.0008,\,0.0267,\,0.9725)$.

**Step 2 — Objective at $\beta=1$.** $\ \mathbb E_{\pi^*}[r]=0.292(1)+0.529(2)=1.350$. $\ D_{\text{KL}}(\pi^*\|\pi_{\text{ref}}) = 0.179\ln\frac{0.179}{0.5}+0.292\ln\frac{0.292}{0.3}+0.529\ln\frac{0.529}{0.2} = -0.184-0.008+0.515 = 0.323$. Objective: $1.350-0.323=1.027$ (which equals $\beta\log Z=\ln2.793=1.027$, as the derivation in the notes predicts).

**Step 3 — Limits.** As $\beta\to\infty$, $e^{r/\beta}\to1$ and $\pi^*\to\pi_{\text{ref}}$: no change. As $\beta\to0$, the weight of the highest-reward response dominates and $\pi^*$ concentrates on it: the policy becomes deterministic, and the $\beta=0.25$ case is already close. Between the two, $\beta$ sets how much of the reference's diversity is traded for reward — the same trade-off seen in the chapter's RLHF runs.

**Key concept**

The KL-regularised objective is solved in closed form by the reference distribution reweighted by $e^{r/\beta}$; the optimal objective value is $\beta\log Z$. RLHF approximates this policy by sampling because $Z$ is a sum over all responses; DPO exploits the formula without ever computing $Z$.

### Exercise 13.5 — The DPO loss and its gradient weights

For a comparison $(x,y_w,y_l)$, the policy and reference log-probabilities are $\log\pi_\theta(y_w\mid x)=-10$, $\log\pi_{\text{ref}}(y_w\mid x)=-12$, $\log\pi_\theta(y_l\mid x)=-9$, $\log\pi_{\text{ref}}(y_l\mid x)=-8$; $\beta=0.1$.

1. Compute the implicit rewards (without the $Z$ term) and the DPO loss of this pair.
2. Compute the weight $\sigma\big(\hat r_\theta(y_l)-\hat r_\theta(y_w)\big)$ that multiplies the gradient, and interpret it.
3. Why is the partition function $Z(x)$ absent from the loss, and what would happen if the two responses belonged to different prompts?

**Step 1 — Implicit rewards and loss.** $\ \hat r_\theta(y_w)=0.1\big(-10-(-12)\big)=+0.2$, $\ \hat r_\theta(y_l)=0.1\big(-9-(-8)\big)=-0.1$. Margin $0.3$; loss $-\log\sigma(0.3)=-\log0.5744=0.554$.

**Step 2 — Weight.** $\ \sigma(-0.1-0.2)=\sigma(-0.3)=0.426$. The policy already ranks the pair correctly (margin $+0.3$) but not confidently, so the pair still contributes $43\%$ of the maximal weight. A pair with margin $+3$ would have weight $\sigma(-3)=0.047$ and be almost ignored; a pair ranked the wrong way, margin $-3$, would have weight $0.953$. DPO concentrates its updates on the comparisons the policy currently gets wrong, like the reward-model loss of Exercise 13.2.

**Step 3 — The partition function.** $\beta\log Z(x)$ appears in the implicit reward of both responses *to the same prompt* and cancels in their difference. If the two responses answered different prompts $x\ne x'$, the difference would contain $\beta\log Z(x)-\beta\log Z(x')$, an unknown constant, and the difference of implicit rewards would no longer equal the Bradley–Terry logit, so the loss would be wrong by an unknown offset: DPO needs comparisons between responses to the **same** prompt.

**Key concept**

The DPO loss is the Bradley–Terry loss with the reward replaced by $\beta\log\frac{\pi_\theta}{\pi_{\text{ref}}}$; its gradient pushes $y_w$ up and $y_l$ down with a weight equal to the probability that the implicit reward ranks the pair wrongly, and it works only because the intractable $Z(x)$ cancels within a prompt.

### Exercise 13.6 — Group-relative advantages

For one prompt, GRPO samples a group of $G=6$ responses whose verifier rewards are $(1,\,0,\,0,\,1,\,0,\,0)$.

1. Compute the group mean and the (unbiased) standard deviation, and the advantages $\hat A_i$.
2. What are the advantages for a group with rewards $(1,1,1,1,1,1)$? And $(0,0,0,0,0,0)$? What does the policy learn from these groups?
3. Compare the group mean with the critic of A2C/PPO: what does it estimate, and what does it cost?

**Step 1 — Advantages.** Mean $=2/6=0.333$. Unbiased variance $=\frac{1}{5}\big[2(1-0.333)^2+4(0-0.333)^2\big]=\frac{1}{5}\big[0.889+0.444\big]=0.267$, std $=0.516$.

$\displaystyle \hat A_{\text{correct}}=\frac{1-0.333}{0.516}=+1.29,\qquad \hat A_{\text{wrong}}=\frac{0-0.333}{0.516}=-0.65.$

The two correct responses are pushed up strongly, the four wrong ones pushed down more gently; the advantages sum to zero within the group.

**Step 2 — Degenerate groups.** All ones or all zeros: mean $=r$, every $r_i-\text{mean}=0$, so $\hat A_i=0$ (the small constant in the denominator avoids a division by zero). These groups contribute **nothing** to the update: a prompt the model always solves, or never solves, carries no information about which responses are better. In practice, prompts of the right difficulty, solved sometimes, are the ones that drive learning — one reason curricula of problems are chosen to have intermediate solve rates.

**Step 3 — Group mean as critic.** The group mean $\frac1G\sum_i r_i$ is a Monte Carlo estimate of $v(x)=\mathbb E_{y\sim\pi}[r(x,y)]$, the value of the prompt under the current policy — exactly what the learned critic of Chapters 09–11, evaluated at the prompt, would provide as a baseline. It costs $G$ generations per prompt instead of a second network as large as the policy, is unbiased, has no approximation error, but is noisier (variance $\approx\mathrm{Var}[r]/G$) and does not provide per-token values, so the advantage is the same for every token of a response.

**Key concept**

GRPO replaces the learned critic by the mean reward of a group of samples for the same prompt: an unbiased, model-free baseline that costs generations instead of parameters. Standardising within the group makes the update scale-free and zeroes the contribution of prompts that are always or never solved.

### Exercise 13.7 — The per-token KL estimator of GRPO

GRPO penalises, at every token, $\hat D=\rho-\log\rho-1$ with $\rho=\pi_{\text{ref}}(y_t\mid\cdot)/\pi_\theta(y_t\mid\cdot)$.

1. Evaluate $\hat D$ for $\rho=0.5$, $1$, $2$.
2. Show that $\hat D\ge0$ with equality only at $\rho=1$.
3. Show that $\mathbb E_{y_t\sim\pi_\theta}[\hat D]=D_{\text{KL}}\big(\pi_\theta(\cdot\mid s)\,\|\,\pi_{\text{ref}}(\cdot\mid s)\big)$, and compare with the estimator $-\log\rho$.

**Step 1 — Values.** $\rho=0.5$: $0.5+0.693-1=0.193$; $\ \rho=1$: $0$; $\ \rho=2$: $2-0.693-1=0.307$. The penalty grows whether the policy has made the token more likely than the reference ($\rho<1$) or less likely ($\rho>1$).

**Step 2 — Non-negativity.** $f(\rho)=\rho-\log\rho-1$ has $f'(\rho)=1-1/\rho$, zero at $\rho=1$, negative before and positive after, and $f(1)=0$: so $f\ge0$ with equality only at $\rho=1$ (equivalently, $\log\rho\le\rho-1$).

**Step 3 — Unbiasedness.** Under $y_t\sim\pi_\theta$: $\ \mathbb E[\rho]=\sum_a\pi_\theta(a)\frac{\pi_{\text{ref}}(a)}{\pi_\theta(a)}=1$, so $\mathbb E[\rho-1]=0$ and $\mathbb E[\hat D]=\mathbb E\big[-\log\rho\big]=\mathbb E\Big[\log\frac{\pi_\theta}{\pi_{\text{ref}}}\Big]=D_{\text{KL}}(\pi_\theta\|\pi_{\text{ref}})$. The naive estimator $-\log\rho$ has the same expectation but can be negative and has larger variance when the two policies are close; $\hat D$ adds the zero-mean term $\rho-1$ as a control variate — the same trick as the approximate KL of Chapter 11, with the ratio inverted (when $\rho$ can be large, far from the reference, the variance of $\hat D$ grows instead). Note that unbiasedness concerns the *value* of the KL: differentiating $\hat D$ with respect to $\theta$ with the sampled tokens held fixed gives the gradient of the reversed divergence $D_{\text{KL}}(\pi_{\text{ref}}\|\pi_\theta)$, which also vanishes at $\pi_\theta=\pi_{\text{ref}}$ and is what GRPO actually descends.

**Key concept**

$\rho-\log\rho-1$ is a non-negative, unbiased per-token estimator of the KL divergence from the reference, with low variance near the reference, which GRPO subtracts directly in the loss instead of adding it to the reward.

### Exercise 13.8 — Reward over-optimisation (integrative)

In the toy language of the chapter, the hidden human score is $1$ per "cat" and $0.5$ per "happy" in a grammatical sentence, and $0$ otherwise. Suppose the reward model has learned $r_\phi(y)=1.0\cdot\#\text{cat}+0.5\cdot\#\text{happy}-0.2$ for *every* string, grammatical or not.

1. What is the best grammatical sentence and its scores under the human and under $r_\phi$?
2. Give a string of at most nine words that $r_\phi$ scores higher than any sentence, and its human score.
3. The KL penalty of a response is $\beta\log\frac{\pi_\theta(y)}{\pi_{\text{ref}}(y)}$. For the hacked string, $\log\pi_{\text{ref}}(y)\approx-40$ (nine improbable tokens), while for the best sentence $\log\pi_{\text{ref}}(y)\approx-9$; a collapsed policy gives either string probability $\approx1$. For which values of $\beta$ does the regularised objective prefer the sentence to the hacked string?

**Step 1 — Best sentence.** Two nouns and two adjectives at most: "the happy cat likes the happy cat ." — human $2+1=3$, reward model $3-0.2=2.8$. On sentences the proxy and the truth agree.

**Step 2 — Hacking.** "cat cat cat cat cat cat cat cat cat" (nine words): $r_\phi=9-0.2=8.8$, human $0$. "happy cat happy cat happy cat happy cat happy" gives $r_\phi=4+2.5-0.2=6.3$. The proxy was fitted on sentences only; outside them it extrapolates linearly, and the optimiser goes where the extrapolation is largest.

**Step 3 — The role of $\beta$.** With $\pi_\theta(y)\approx1$, the penalty is $\beta(0-\log\pi_{\text{ref}}(y))=-\beta\log\pi_{\text{ref}}(y)$. Regularised objective: sentence $2.8-9\beta$; hacked string $8.8-40\beta$. The sentence wins when $2.8-9\beta>8.8-40\beta$, i.e. $31\beta>6$, $\beta>0.19$. Below that the hack pays; above it the penalty for leaving the reference's language outweighs the proxy's bonus — which is what the chapter's runs with $\beta=0$ and $\beta=0.3$ showed. (With $\beta$ much larger the policy would not even move to the best sentence. If the reference samples the grammar uniformly, as the chapter's corpus does (four nouns, an adjective present with probability $1/2$ and chosen among four), each noun is a cat with probability $1/4$ and each adjective slot is "happy" with probability $1/2 \cdot 1/4$, so $\mathbb E_{\pi_{\text{ref}}}[r_\phi]=2\cdot\tfrac14+0.5\cdot 2\cdot\tfrac18-0.2=0.425$ at zero KL; the sentence $2.8-9\beta$ falls below it for $\beta>0.26$.)

**Key concept**

A learned reward is a proxy that is accurate on the data it was fitted to and arbitrary elsewhere; unconstrained optimisation finds the elsewhere. The KL penalty prices the departure from the reference, and $\beta$ must be large enough to make the departure not worth the proxy's bonus, yet small enough to allow genuine improvement — Goodhart's law, managed with a trust region.